In [1]:
from pathlib import Path
import duckdb
import pandas as pd
from metrics import apk, mapk

DATA = Path("data")
con = duckdb.connect()
con.execute(f"CREATE VIEW tx AS SELECT * FROM '{(DATA / 'sample_transactions.parquet').as_posix()}'")

# Metric sanity checks
print(round(apk(["A", "B"], ["A", "X", "B"]), 3), "expected 0.833")
print(apk(["A"], ["X", "Y"]), "expected 0.0")
print(apk(["A"], ["A", "A", "A"]), "expected 1.0 (repeat guesses don't add)")
print(mapk({"c1": ["A"], "c2": ["B"]}, {"c1": ["A"]}), "expected 0.5 (c2 missing = 0)")

0.833 expected 0.833
0.0 expected 0.0
1.0 expected 1.0 (repeat guesses don't add)
0.5 expected 0.5 (c2 missing = 0)


In [2]:
labels = con.execute(f"SELECT customer_id, bought FROM '{(DATA / 'labels_valid.parquet').as_posix()}'").df()
actual = {c: list(b) for c, b in zip(labels["customer_id"], labels["bought"])}
print(f"Validation customers: {len(actual):,}")

def top_items(first_w, n_weeks, k=12):
    return con.execute(f"""
        SELECT article_id
        FROM tx
        WHERE w BETWEEN {first_w} AND {first_w + n_weeks - 1}
        GROUP BY article_id
        ORDER BY COUNT(DISTINCT customer_id) DESC, article_id
        LIMIT {k}
    """).df()["article_id"].tolist()

results = []
for n in [1, 2, 4, 8]:
    top = top_items(first_w=2, n_weeks=n)
    preds = {c: top for c in actual}
    results.append({"window_weeks": n, "MAP@12": round(mapk(actual, preds), 5)})

print(pd.DataFrame(results).to_string(index=False))

Validation customers: 14,471
 window_weeks  MAP@12
            1 0.00680
            2 0.00686
            4 0.00477
            8 0.00459


In [3]:
import numpy as np

top2 = top_items(first_w=2, n_weeks=2)
per_customer = np.array([apk(a, top2) for a in actual.values()])

rng = np.random.default_rng(42)
boot = [per_customer[rng.integers(0, len(per_customer), len(per_customer))].mean()
        for _ in range(1000)]
low, high = np.percentile(boot, [2.5, 97.5])

print(f"Popularity (2 weeks) MAP@12 = {per_customer.mean():.5f}")
print(f"95% bootstrap range:          {low:.5f} to {high:.5f}")
print(f"Customers with any hit:       {(per_customer > 0).mean():.2%}")

Popularity (2 weeks) MAP@12 = 0.00686
95% bootstrap range:          0.00605 to 0.00767
Customers with any hit:       5.35%


## Popularity baseline

The simplest thing I could think of: give every customer the same 12 best-selling items. "Best-selling" means the most different buyers, not the most rows, so a few bulk buyers can't push an item up.

I tried four time windows on the validation week. The last 2 weeks scored best (MAP@12 = 0.00686), with 1 week almost identical (0.00680). Going further back made it worse: 0.00477 for 4 weeks and 0.00459 for 8. That fits what I saw in the EDA. In fashion, last month's best-sellers are already old.

0.00686 looks tiny, but only about 5% of customers bought even one of these 12 items. That's the reality of this problem.

To see how much the score moves by chance, I resampled the validation customers 1,000 times. The 95% range was 0.00605 to 0.00767. So the 1-week vs 2-week difference is just noise. Any model I build later has to clear this range by a real margin before I call it better.